In [ ]:
import jax
import jax.numpy as jnp


class LinearLayer:
    def __init__(self, in_dim, out_dim):
        self.in_dim = in_dim
        self.out_dim = out_dim

    def init(self, key):
        W = jax.random.normal(key, (self.in_dim, self.out_dim)) \
            * jnp.sqrt(2.0 / self.in_dim)
        b = jnp.zeros((self.out_dim,), dtype=jnp.float32)
        return {"W": W.astype(jnp.float32), "b": b}

    def forward(self, params, x):
        return x @ params["W"] + params["b"]

    # Manual backward (given upstream gradient)
    def backward(self, params, x, grad_output):
        """
        x: input to this layer (N, in_dim)
        grad_output: dL/dZ (N, out_dim)
        """

        dW = x.T @ grad_output
        db = jnp.sum(grad_output, axis=0)
        grad_input = grad_output @ params["W"].T

        grads = {"W": dW, "b": db}
        return grads, grad_input

In [ ]:
class ReLU:
    def forward(self, x):
        return jnp.maximum(x, 0.0)

    def backward(self, x, grad_output):
        return grad_output * (x > 0).astype(x.dtype)


# Setup
key = jax.random.PRNGKey(0)

layer1 = LinearLayer(784, 128)
layer2 = LinearLayer(128, 10)
relu = ReLU()

k1, k2 = jax.random.split(key)
params1 = layer1.init(k1)
params2 = layer2.init(k2)

# Forward
def forward(params1, params2, x):
    z1 = layer1.forward(params1, x)
    a1 = relu.forward(z1)
    z2 = layer2.forward(params2, a1)
    return z1, a1, z2

# Manual backward (softmax + CE top gradient assumed computed)
def backward(params1, params2, x, z1, a1, grad_logits):

    # Layer 2
    grads2, grad_a1 = layer2.backward(params2, a1, grad_logits)

    # ReLU
    grad_z1 = relu.backward(z1, grad_a1)

    # Layer 1
    grads1, _ = layer1.backward(params1, x, grad_z1)

    return grads1, grads2

In [2]:
import jax
import jax.numpy as jnp

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# -------------------------
# Data (Torchvision -> NumPy)
# -------------------------
def load_mnist_torch(split, batch_size, shuffle=True, seed=0, root="./data"):
    """
    Yields (x, y) where:
      x: float32 (B, 784) in [0,1]
      y: int32   (B,)
    """
    is_train = (split == "train")

    tfm = transforms.ToTensor()  # gives (1,28,28) float32 in [0,1]
    ds = datasets.MNIST(root=root, train=is_train, download=True, transform=tfm)

    g = torch.Generator()
    g.manual_seed(seed)

    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
        num_workers=0,
        generator=g if shuffle else None,
        pin_memory=False,
    )

    for x, y in loader:
        x = x.view(x.shape[0], -1).numpy().astype("float32")
        y = y.numpy().astype("int32")
        yield x, y

# -------------------------
# Model setup (reuse classes from above)
# -------------------------
layer_sizes = [784, 128, 64, 10]
linear_layers = [
    LinearLayer(in_dim, out_dim)
    for in_dim, out_dim in zip(layer_sizes[:-1], layer_sizes[1:])
]
relu = ReLU()


def init_params(key):
    keys = jax.random.split(key, len(linear_layers))
    return tuple(layer.init(k) for layer, k in zip(linear_layers, keys))


# -------------------------
# Forward
# -------------------------
def forward(params, X):
    hidden_caches = []
    A = X

    # Hidden linear + ReLU blocks
    for i, layer in enumerate(linear_layers[:-1]):
        Z = layer.forward(params[i], A)
        hidden_caches.append({"A_in": A, "Z": Z})
        A = relu.forward(Z)

    # Final linear layer
    logits = linear_layers[-1].forward(params[-1], A)
    A_last = A
    return logits, hidden_caches, A_last


# -------------------------
# Softmax + Cross-Entropy (stable)
# -------------------------
def softmax(logits):
    logits = logits - jnp.max(logits, axis=1, keepdims=True)
    exp = jnp.exp(logits)
    return exp / jnp.sum(exp, axis=1, keepdims=True)


def xent_loss_and_probs(logits, y):
    P = softmax(logits)
    N = logits.shape[0]
    logp = jnp.log(P[jnp.arange(N), y] + 1e-12)
    loss = -jnp.mean(logp)
    return loss, P


# -------------------------
# Manual backward
# -------------------------
def backward(params, X, y, logits, hidden_caches, A_last, P):
    N = X.shape[0]
    C = logits.shape[1]

    Y = jax.nn.one_hot(y, C)
    dLogits = (P - Y) / N

    grads = [None] * len(linear_layers)

    # Final linear layer
    grads[-1], dA = linear_layers[-1].backward(params[-1], A_last, dLogits)

    # Hidden layers: ReLU backward then Linear backward
    for i in range(len(linear_layers) - 2, -1, -1):
        cache = hidden_caches[i]
        dZ = relu.backward(cache["Z"], dA)
        grads[i], dA = linear_layers[i].backward(params[i], cache["A_in"], dZ)

    return tuple(grads)


# -------------------------
# SGD update
# -------------------------
def sgd_step(params, grads, lr):
    return jax.tree.map(lambda p, g: p - lr * g, params, grads)


# -------------------------
# Metrics
# -------------------------
def accuracy_from_logits(logits, y):
    preds = jnp.argmax(logits, axis=1)
    return jnp.mean((preds == y).astype(jnp.float32))


# -------------------------
# One training step (JIT)
# -------------------------
@jax.jit
def train_step(params, X, y, lr):
    logits, hidden_caches, A_last = forward(params, X)
    loss, P = xent_loss_and_probs(logits, y)
    acc = accuracy_from_logits(logits, y)
    grads = backward(params, X, y, logits, hidden_caches, A_last, P)
    params = sgd_step(params, grads, lr)
    return params, loss, acc


@jax.jit
def eval_step(params, X, y):
    logits, _, _ = forward(params, X)
    loss, _ = xent_loss_and_probs(logits, y)
    acc = accuracy_from_logits(logits, y)
    return loss, acc


# -------------------------
# Train loop
# -------------------------
key = jax.random.PRNGKey(0)
params = init_params(key)

batch_size = 256
lr = 0.1
epochs = 5

for epoch in range(1, epochs + 1):
    tr_losses, tr_accs = [], []
    for Xb, yb in load_mnist_torch("train", batch_size=batch_size, shuffle=True, seed=epoch):
        params, loss, acc = train_step(params, jnp.array(Xb), jnp.array(yb), lr)
        tr_losses.append(float(loss))
        tr_accs.append(float(acc))

    te_losses, te_accs = [], []
    for Xb, yb in load_mnist_torch("test", batch_size=1000, shuffle=False, seed=0):
        loss, acc = eval_step(params, jnp.array(Xb), jnp.array(yb))
        te_losses.append(float(loss))
        te_accs.append(float(acc))

    print(
        f"[epoch {epoch:02d}] "
        f"train loss={sum(tr_losses)/len(tr_losses):.4f} train acc={sum(tr_accs)/len(tr_accs)*100:.2f}% | "
        f"test loss={sum(te_losses)/len(te_losses):.4f} test acc={sum(te_accs)/len(te_accs)*100:.2f}%"
    )

100%|██████████| 9.91M/9.91M [00:11<00:00, 901kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 992kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 826kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.60MB/s]


[epoch 01] train loss=0.5578 train acc=84.17% | test loss=0.3032 test acc=90.94%
[epoch 02] train loss=0.2659 train acc=92.30% | test loss=0.2683 test acc=92.20%
[epoch 03] train loss=0.2098 train acc=93.97% | test loss=0.1867 test acc=94.55%
[epoch 04] train loss=0.1772 train acc=94.96% | test loss=0.1644 test acc=95.20%
[epoch 05] train loss=0.1533 train acc=95.65% | test loss=0.1518 test acc=95.23%
